# Flexible distributions: wind and temperature

Run the cells from top to bottom. When the upload button appears,
select **`estacao_A001_diario.csv`**.

We will fit the GLD-FKML and compare it with classical distributions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wmpjrufg/distribuicoes-flexiveis/blob/main/notebooks/distribuicoes_flexiveis_en.ipynb)

In [ ]:
%pip install -q pyglam==1.1.0

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pyglam import GlamFKML
from google.colab import files

## 1. Upload and read the data

The CSV contains `data` (date), `vento_ms` (wind speed in m/s)
and `temperatura_c` (temperature in °C). The column names stay unchanged.

In [ ]:
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
df['data'] = pd.to_datetime(df['data'])
df.head()

## 2. A function to compare fits

The plot shows the histogram and densities. The table shows the Kolmogorov–Smirnov
distance (`Dn`) and deviations at the 5%, 50% and 95% quantiles.
Deviation = model quantile − data quantile. Values closer to zero indicate better agreement.
We use `Dn` only as a descriptive measure, not as a hypothesis test.

In [ ]:
def compare(x, models, xlabel):
    p = np.array([0.05, 0.50, 0.95])
    empirical = np.quantile(x, p)
    grid = np.linspace(x.min(), x.max(), 400)
    rows = []
    plt.figure(figsize=(7, 4))
    plt.hist(x, bins=30, density=True, alpha=0.3, color='gray')
    for name, model in models.items():
        plt.plot(grid, model.pdf(grid), label=name)
        delta = model.ppf(p) - empirical
        rows.append([name, stats.ks_1samp(x, model.cdf).statistic, *delta])
    plt.xlabel(xlabel)
    plt.ylabel('Density')
    plt.legend()
    plt.show()
    return pd.DataFrame(rows, columns=['Model', 'Dn', 'Δ5%', 'Δ50%', 'Δ95%']).round(3)

## 3. Daily mean wind speed

We use available observations without filling gaps. The GLD is fitted by
the method of moments; Rayleigh and Weibull use maximum likelihood,
with their origin fixed at zero.

In [ ]:
wind = df['vento_ms'].dropna().to_numpy()
fit_wind = GlamFKML().fit_lambdas(wind, n_starts=15, seed=0)
gld_wind = GlamFKML(*fit_wind.x)
_, rayleigh_scale = stats.rayleigh.fit(wind, floc=0)
shape, _, scale = stats.weibull_min.fit(wind, floc=0)

print('n =', len(wind))
print('λ =', fit_wind.x.round(4))
compare(wind, {
    'Rayleigh': stats.rayleigh(scale=rayleigh_scale),
    'Weibull': stats.weibull_min(shape, scale=scale),
    'GLD-FKML': gld_wind,
}, 'Wind speed (m/s)')

### Support matters too

A good moment fit does not guarantee that the model includes all observations.
Let us count how many days fall outside the fitted GLD bounds.

In [ ]:
l1, l2, l3, l4 = fit_wind.x
lower = l1 - 1 / (l2 * l3) if l3 > 0 else -np.inf
upper = l1 + 1 / (l2 * l4) if l4 > 0 else np.inf
outside = np.sum((wind < lower) | (wind > upper))
print(f"Support: {lower:.3f} to {upper:.3f} m/s")
print(f"Days outside support: {outside} ({100 * outside / len(wind):.2f}%)")

## 4. Daily mean temperature in July

We select July to reduce mixing across seasons.
We compare the GLD-FKML with the normal distribution.

In [ ]:
temperature = df.loc[df['data'].dt.month == 7, 'temperatura_c'].dropna().to_numpy()
fit_temperature = GlamFKML().fit_lambdas(temperature, n_starts=15, seed=0)
gld_temperature = GlamFKML(*fit_temperature.x)
mean, std = stats.norm.fit(temperature)

print('n =', len(temperature))
print('λ =', fit_temperature.x.round(4))
compare(temperature, {
    'Normal': stats.norm(mean, std),
    'GLD-FKML': gld_temperature,
}, 'Temperature (°C)')

## Questions to explore

- Which model has the smallest wind quantile deviations?
- What does it mean for the GLD to exclude some days from its support?
- Does the improved temperature fit justify four parameters instead of two?

These are descriptive fits. Daily observations are temporally dependent,
and this short example does not assess sampling uncertainty.